In [1]:
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity in the beginning 
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

--------------------------------------------------------------------------
Ignoring value for oob_tcp_if_exclude on worker6063 (10.250.112.0/20: Did not find interface matching this subnet).
(You can safely ignore this message.)
--------------------------------------------------------------------------


# GW Electronic Structure with CoQuí

In this notebook, we explore CoQuí’s capabilities for performing GW many-body perturbation theory calculations, which cover all three phases of the CoQuí workflow shown in the figure below. Along the way, we will use GW as a concrete example to illustrate the overall structure of a many-body calculation in CoQuí. The resulting GW script can serve as a **reusable template** for other electronic structure solvers available in CoQuí.

**What you’ll learn:**
1. How to **build a many-body Hamiltonian** for electronic structure solver in CoQuí (applicable beyond GW)
2. The key parameters controlling a GW calculation in CoQuí
4. How to visualize the GW band structure and compare it with DFT results

**Stretch goal**: Experiment with different GW variants and observe their impact on the results. 

## 🔹 GW Workflow at a glance

<figure style="text-align: center;">
  <img src="./images/coqui_workflow_gw.png" alt="GW workflow in CoQuí" width="60%">
  <figcaption><em>Figure&nbsp;1:</em> GW workflow in CoQuí.</figcaption>
</figure>

At a high level, the GW workflow in CoQuí proceeds as follows:

1. **Problem Setup**
   - Build a mean-field object (`coqui.Mf`) containing the single-particle information (structure, k-mesh, bands, orbitals) (see the [previous notebook](01_dft_to_coqui_converter.ipynb)).
   - Construct the Coulomb Hamiltonian in a **compressed** format (THC or Cholesky) to reduce the cost of later many-body calculations while maintaining accuracy.
2. **Simulation (GW)**
   - Solve the many-body problem within the GW approximation.
3. **Post-Processing**:
   - Interpolate the GW electronic structure for fine k-resolution.
   - Analytically continue imaginary-axis quantities to for physical quantities such as band structures and local density of states.
  
Translating these phases into code yields the following real-world python script for the GW calculation of silicon:
```python
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity 
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

# --- Phase 1: Problem setup ---
# Mean-field (DFT) description of the target system
mf_params = {
    "prefix": "si", 
    "outdir": "data/qe_inputs/si/out", 
    "nbnd": 10
}
mf = coqui.make_mf(coqui_mpi, mf_params, "qe")

# Construct Coulomb Hamiltonian (in a compressed THC format)
thc_params = {
    "ecut": 35,
    "thresh": 1e-2
}
thc = coqui.make_thc_coulomb(mf, thc_params)

# --- Phase 2: Simulation (GW) ---
gw_params = {
    "restart": False,
    "output": "gw",
    "beta": 500,
    "wmax": 1.8,
    "iaft_prec": "medium",
    "niter": 2,
}
coqui.run_gw(gw_params, h_int = thc)

# --- Phase 3: Post-processing ---
# (plots / analysis shown later in the notebook)
```
Set aside the API details for now. Note the simplicity of the code structure and the clear separation between phases. We will now break this example down, and walk though each phase step by step. 

> 💡**Tip**  
> This three-phases structure is not specific to GW; it is common to all electronic structure methods in CoQuí. In particular, the problem setup in Phase 1 is shared across methods, while only the function calls in Phase 2 and 3 need to be changed to match the chosen method. 


### 🔹 Phase 1: Problem setup — what's happening?

<figure style="text-align: center;">
  <img src="./images/coqui_problem_setup.png" alt="Phase 1 of the many-body calculation workflow in CoQuí" width="60%">
  <figcaption><em>Figure&nbsp;2:</em> Phase 1 of the many-body calculation workflow in CoQuí. </figcaption>
</figure>


```python
from mpi4py import MPI
import coqui

# Create CoQui MPI handler and set logging verbosity 
coqui_mpi = coqui.MpiHandler()
coqui.set_verbosity(coqui_mpi, output_level=1)

# --- Phase 1: Problem setup ---
# Mean-field (DFT) description of the target system
mf_params = {
    "prefix": "si", 
    "outdir": "data/qe_inputs/si/out", 
    "nbnd": 10
}
mf = coqui.make_mf(coqui_mpi, mf_params, "qe")

# Construct THC Coulomb Hamiltonian (compressed ERIs)
thc_params = {
    "ecut": 35,
    "thresh": 1e-2
}
thc = coqui.make_thc_coulomb(mf, thc_params)
```

In this phase, the new step is `coqui.make_thc_coulomb(...)`, which builds the Coulomb Hamiltonian in Tensor HyperContraction (THC) form — a compact representation of the interaction part of the many-body problem. 

**Step‑by‑step:**  
1. `coqui.make_mf` — Reads the DFT outputs and returns an `coqui.Mf` object encapsulating all data needed for the non-interacting Hamiltonian. See the [previous notebook](01_dft_to_coqui_converter.ipynb) for a quick recap or the [Mean Field notebook](CoQui-tutorials/01a-Mean_Field.ipynb) for details.
2. `coqui.make_thc_coulomb` — Builds the THC Coulomb Hamiltonian using the orbitals in `Mf` and returns a `ThcCoulomb` objcet, used by electronic structure solvers to access the Coulomb Hamiltonian. See the [THC Coulomb notebook](CoQui-tutorials/02a-THC_Coulomb_Hamiltonian.ipynb) for details.

**Accuracy-relevant parameters:**
- `thc_params["ecut"]` — plane‑wave cutoff (in Hartree unit) for building the Coulomb Hamiltonian. 
- `thc_params["thresh"]` — THC accuracy threshold (smaller → more accurate, more expensive).

> 💡**SUMMARY**  
> The conceptual road map is to construct: **MpiHandler → Mean-Field → THC Hamiltonian**. That’s the same pattern you’ll reuse for other CoQuí methods.

> 💡**TIP**  
> This resulting `ThcCoulomb` object is universal across all the electronic structure methods in CoQuí: the same THC Hamiltonian can be passed to HF, GW, downfolding routines, and other solvers without modifications. 

### ▶️ Hands-on 1: Build Your First *Ab Initio* THC Coulomb Hamiltonian 

1. Copy and paste the Phase 1 section from the GW Python script into a new cell, and run it.
2. Observe the log output. How many **interpolating points** do you see?

From the log, answer:
- What FFT mesh / number of plane waves were used at `ecut=35`?
- What THC rank / number of interpolating points was chosen (if reported)?
- What compression factor (approx.) do you infer (e.g., ERI size in its full vs THC representation)?

> 💡Tip  
> The THC approximates the four-index Coulomb integrals into
> $$
V^{\textbf{k}_1\textbf{k}_2\textbf{k}_3\textbf{k}_4} _{abcd} \approx \sum^{N_{\mu}}_{\mu\nu}
X^{\textbf{k}_1*} _{\mu a} X^{\textbf{k}_2} _{\mu b}V^{\textbf{q}} _{\mu\nu} 
X^{\textbf{k}_3*} _{\nu c}X^{\textbf{k}_4} _{\nu d}
  $$
> where $\mu$, $\nu$ label the THC auxiliary basis functions.  
> The **compression ratio** can be defined as the ratio of memory required for the full $V$ versus its THC form $V_{\mathrm{THC}}$. 

In [2]:
# --- Phase 1: Problem setup ---
# Mean-field (DFT) description of the target system
mf_params = {
    "prefix": "si", 
    "outdir": "data/qe_inputs/si/out", 
    "nbnd": 10
}
mf = coqui.make_mf(coqui_mpi, mf_params, "qe")
# Construct THC Coulomb Hamiltonian (compressed ERIs)
thc_params = {
    "ecut": 35,
    "thresh": 1e-2
}
thc = coqui.make_thc_coulomb(mf, thc_params)

  Quantum ESPRESSO reader
  -----------------------
  Number of spins                = 1
  Number of polarizations        = 1
  Number of bands                = 10
  Monkhorst-Pack mesh            = (2,2,2)
  K-points                       = 8 total, 4 in the IBZ
  Number of electrons            = 8.0
  Electron density energy cutoff = 200.000 a.u. | FFT mesh = (36,36,36)
  Wavefunction energy cutoff     = 28.496 a.u. | FFT mesh = (17,17,17), Number of PWs = 1989


╔═╗╔═╗╔═╗ ╦ ╦╦  ╔╦╗┬ ┬┌─┐╔═╗┌─┐┬ ┬┬  ┌─┐┌┬┐┌┐ 
║  ║ ║║═╬╗║ ║║   ║ ├─┤│  ║  │ ││ ││  │ ││││├┴┐
╚═╝╚═╝╚═╝╚╚═╝╩   ╩ ┴ ┴└─┘╚═╝└─┘└─┘┴─┘└─┘┴ ┴└─┘

  Algorithm                       = ISDF
  THC integrals access            = incore
  Found precomputed THC integrals = false
  --> CoQuí will compute THC integrals.

  ERI::thc Computation Details
  ----------------------------
  Energy cutoff                = 35.0 a.u. | FFT mesh = (19,19,19), Number of PWs = 2685
  Threshold                    = 0.01

*******************************

### 🔹 Phase 2: Simulation (GW) — what's happening?

<figure style="text-align: center;">
  <img src="./images/coqui_simulation_gw.png" alt="Phase 2 of the GW workflow in CoQuí" width="60%">
  <figcaption><em>Figure&nbsp;3:</em> Phase 2 of the GW workflow in CoQuí. </figcaption>
</figure>


In Phase 2, we pass the `ThcCoulomb` object from Phase 1 to CoQuí’s GW solver, together with GW-specific parameters. The solver runs one or more **Dyson self-consistent field (SCF) iterations** to determine G, W, and Σ self-consistently, all within a single function call:

```python
# --- Phase 2: Simulation (GW) ---
gw_params = {
    "restart": False,    # read prior GW state if available
    "output":  "gw",     # label/prefix for outputs
    "niter":   2,        # number of Dyson–SCF iterations
    "beta":    500,      # inverse temperature (Ha^{-1})
    "wmax":    1.8,      # frequency cutoff for the IR basis
    "iaft_prec": "medium", # imaginary-axis FFT/IR accuracy target
}
coqui.run_gw(gw_params, h_int = thc)
```
`coqui.run_gw` takes only two inputs — the interacting Hamiltonian (`ThcCoulomb`) and the GW parameters (`dict`) — and handles the rest of the simulation internally. Behind this single call is the **Dyson SCF loop**, illustrated below.

<figure style="text-align: center;">
  <img src="./images/dyson_scf.png" alt="Dyson SCF" width="60%">
  <figcaption><em>Figure&nbsp;3:</em> CoQuí Dyson-SCF driver  </figcaption>
</figure>

**One Dyson SCF iteration**:
1. **Initialize $G(i\omega_{n})$** - use the current self-energy and adjust chemical potential $\mu$ to preserve the electron number. 
3. **CoQuí THC-HF** - compute the self-energy at infinity frequency limit $\Sigma_{\infty} = \Sigma_{\mathrm{HF}}$ via Hartree-Fock. 
4. **CoQuí Screened Coulomb** - calculate the screened interactions $W(\tau)$ via the RPA polarization: $(1 - V\Pi_{\mathrm{RPA}})^{-1}V$. 
5. **CoQuí THC-GW** - evaluate the dynamic GW self-energy $\Sigma(\tau) = G(\tau) W(\tau)$
6. Return to Step 1 if `niter` has not been reached or the convergence threshold is not met. 

> 💡**Note**  
> The initial guess for $\Sigma_{\infty}$ comes from the effective one-body Hamiltonian in `coqui.Mf`. With `niter=1` (and no restart), this produces a single GW update on top of the non-interacting $G$ from the mean-field Hamiltonina. 

### 🔹 Imaginary-axis meshes setup
The Dyson SCF loop evaluates correlation functions G(τ), W(τ), and Σ(τ) on the imaginary axis.
Instead of storing these on dense Matsubara grids which can be extremely costly in CPU time and memory, CoQuí uses a compact Intermediate Representation (IR) basis. The size of this basis, and therefore the cost and accuracy of the simulation, is controlled by three parameters in `gw_params`: 

| Parameter   | Role (intuition)                                                | Effect on IR mesh size                          | Notes                                                                                     |
|-------------|------------------------------------------------------------------|--------------------------------------------------|-------------------------------------------------------------------------------------------|
| `beta`      | Inverse temperature (Ha⁻¹)                                       | Larger `beta` ⇒ **more IR basis functions**      | Set by the physics. Lower $T$ generally results in richer low‑frequency structure  |
| `wmax`      | Frequency cutoff ≈ **energy window (bandwidth)**         | Larger `wmax` ⇒ **more IR basis functions**    | Choose **≥** the spread of input DFT orbital energies.    |
| `iaft_prec` | IR basis precision          | Higher precision ⇒ **more IR basis functions**   | Options: `"low"`, `"medium"`, `"high"`. Higher precision increases cost.    |

> **Rule of thumb**  
> For a fixed beta, increasing lambda and/or iaft_prec → more basis functions → higher accuracy and higher CPU/memory.

### ▶️ Hands-on 2: Read the GW Dyson SCF Log

Run the Phase 2 GW script with `niter = 1` and observe the log output. 

From the log, answer:
1. **Identify the Dyson–SCF workflow**:
   - Can you map the major log sections to the Dyson–SCF steps discussed earlier?  
     (Hint: Look for Hartree–Fock, screened interaction, and GW self-energy blocks.)
2. **IR basis size**:
   - How many frequency points (IR basis functions) were constructed for G and W?
3. **Output verification**:
   - Was an HDF5 file `{output}.mbpt.h5` created after coqui.run_gw completed?
4. **Energy breakdown**:
   - What are the Hartree–Fock, correlation, and total energies for this iteration?  
     (Hint: These values appear near the end of the log. The “correlation” term comes from the dynamic part of the GW self-energy.)
5. **Convergence indicators**:  
   - What are the values of abs max diff of Fock matrix and abs max diff of self-energy?  
     (Hint: These correspond to the maximum norm of the change in the static and dynamic self-energy compared to the previous iteration.)

In [5]:
# --- Phase 2: Simulation (GW) ---
coqui.set_verbosity(coqui_mpi, output_level=1)
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,      
    "beta": 500,
    "wmax": 1.8,
    "iaft_prec": "medium",
}
coqui.run_gw(gw_params, h_int = thc)


╔══════════════════════════════════════════════════════════╗
║ [ WARNING ]                                              ║
║ An existing CoQuí checkpoint HDF5 with the same prefix   ║
║ has been detected even though CoQuí is running in the    ║
║ start-from-scratch mode. --> The old checkpoint will be  ║
║ overwritten. Considering move the old HDF5 or change the ║
║ prefix next time.                                        ║
╚══════════════════════════════════════════════════════════╝


╔═╗╔═╗╔═╗ ╦ ╦╦  ┌┬┐┬ ┬┌─┐┌─┐┌┐┌   ┌─┐┌─┐┌─┐
║  ║ ║║═╬╗║ ║║   ││└┬┘└─┐│ ││││───└─┐│  ├┤ 
╚═╝╚═╝╚═╝╚╚═╝╩  ─┴┘ ┴ └─┘└─┘┘└┘   └─┘└─┘└  

  Maximum iteration number = 1
  Convergence tolerance    = 1e-08
  Checkpoint HDF5          = gw.mbpt.h5
  Restart                  = no
  Number of processors     = 1 cores per node, 1 nodes

  Mesh details on the imaginary axis
  ----------------------------------
  Intermediate Representation
  Beta                   = 500.0 a.u.
  Frequency cutoff       = 2.0 a.u.
  La

### ▶️ Hands-on 2.2: Explore IR Mesh Parameters

In this exercise, you will modify the IR basis parameters — `beta`, `wmax`, and `iaft_prec`, and observe their impact on the imaginary-axis mesh size and solver behavior.

Copy the Phase 2 section of the GW script into a new cell with `niter=1`, and modify `beta`, `wmax` and `iaft_prec` in the following scenarios: 

1. **Lower temperature $T$**:  
   Set `beta` to 10 times its original value (e.g., 500 → 5000), and run the GW calculation. Watch for the following warning in the log:
   > [WARNING] A large IAFT leakage is found: ... > 1e-5, ...   
   > The leakage is roughly the accuracy of the grids used to represent the imaginary axes. Considering increasing "lambda" of the IR/DLR grid.
   
   - Why does larger `beta` make this warning more likely? How would increasing `wmax` help?
   - Does changing `iaft_prec` from `"medium"` to `"high"` help? Why?

2. **Increase `wmax` by a factor of 10** (e.g., 1.8 → 18). 
   Check the log for the reported number of IR basis functions for G and W. How does the increased lambda affect IR basis size? 

3. **Changing `iaft_prec` from `"medium"` to `"low"`**. 
   How many IR basis do we have for G and W? How does this affect the energies? 

> 💡**TIP**  
> Larger `beta` and/or `wmax` → finer resolution in frequency space → more IR basis functions.
Lowering `iaft_prec` can reduce mesh size significantly, but at the cost of accuracy.

In [12]:
# --- Phase 2: Simulation (GW) ---
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,
    "beta": 5000,
    "wmax": 1.8,
    "iaft_prec": "medium",
}
coqui.run_gw(gw_params, h_int = thc)


╔══════════════════════════════════════════════════════════╗
║ [ WARNING ]                                              ║
║ An existing CoQuí checkpoint HDF5 with the same prefix   ║
║ has been detected even though CoQuí is running in the    ║
║ start-from-scratch mode. --> The old checkpoint will be  ║
║ overwritten. Considering move the old HDF5 or change the ║
║ prefix next time.                                        ║
╚══════════════════════════════════════════════════════════╝


╔═╗╔═╗╔═╗ ╦ ╦╦  ┌┬┐┬ ┬┌─┐┌─┐┌┐┌   ┌─┐┌─┐┌─┐
║  ║ ║║═╬╗║ ║║   ││└┬┘└─┐│ ││││───└─┐│  ├┤ 
╚═╝╚═╝╚═╝╚╚═╝╩  ─┴┘ ┴ └─┘└─┘┘└┘   └─┘└─┘└  

  Maximum iteration number = 1
  Convergence tolerance    = 1e-08
  Checkpoint HDF5          = gw.mbpt.h5
  Restart                  = no
  Number of processors     = 1 cores per node, 1 nodes

  Mesh details on the imaginary axis
  ----------------------------------
  Intermediate Representation
  Beta                   = 5000.0 a.u.
  Frequency cutoff       = 2.0 a.u.
  L

In [7]:
# --- Phase 2: Simulation (GW) ---
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,
    "beta": 500,
    "wmax": 18,
    "iaft_prec": "medium",
}
coqui.run_gw(gw_params, h_int = thc)


╔══════════════════════════════════════════════════════════╗
║ [ WARNING ]                                              ║
║ An existing CoQuí checkpoint HDF5 with the same prefix   ║
║ has been detected even though CoQuí is running in the    ║
║ start-from-scratch mode. --> The old checkpoint will be  ║
║ overwritten. Considering move the old HDF5 or change the ║
║ prefix next time.                                        ║
╚══════════════════════════════════════════════════════════╝


╔═╗╔═╗╔═╗ ╦ ╦╦  ┌┬┐┬ ┬┌─┐┌─┐┌┐┌   ┌─┐┌─┐┌─┐
║  ║ ║║═╬╗║ ║║   ││└┬┘└─┐│ ││││───└─┐│  ├┤ 
╚═╝╚═╝╚═╝╚╚═╝╩  ─┴┘ ┴ └─┘└─┘┘└┘   └─┘└─┘└  

  Maximum iteration number = 1
  Convergence tolerance    = 1e-08
  Checkpoint HDF5          = gw.mbpt.h5
  Restart                  = no
  Number of processors     = 1 cores per node, 1 nodes

  Mesh details on the imaginary axis
  ----------------------------------
  Intermediate Representation
  Beta                   = 500.0 a.u.
  Frequency cutoff       = 20.0 a.u.
  L

In [8]:
# --- Phase 2: Simulation (GW) ---
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,
    "beta": 500,
    "wmax": 1.8,
    "iaft_prec": "low",
}
coqui.run_gw(gw_params, h_int = thc)


╔══════════════════════════════════════════════════════════╗
║ [ WARNING ]                                              ║
║ An existing CoQuí checkpoint HDF5 with the same prefix   ║
║ has been detected even though CoQuí is running in the    ║
║ start-from-scratch mode. --> The old checkpoint will be  ║
║ overwritten. Considering move the old HDF5 or change the ║
║ prefix next time.                                        ║
╚══════════════════════════════════════════════════════════╝


╔═╗╔═╗╔═╗ ╦ ╦╦  ┌┬┐┬ ┬┌─┐┌─┐┌┐┌   ┌─┐┌─┐┌─┐
║  ║ ║║═╬╗║ ║║   ││└┬┘└─┐│ ││││───└─┐│  ├┤ 
╚═╝╚═╝╚═╝╚╚═╝╩  ─┴┘ ┴ └─┘└─┘┘└┘   └─┘└─┘└  

  Maximum iteration number = 1
  Convergence tolerance    = 1e-08
  Checkpoint HDF5          = gw.mbpt.h5
  Restart                  = no
  Number of processors     = 1 cores per node, 1 nodes

  Mesh details on the imaginary axis
  ----------------------------------
  Intermediate Representation
  Beta                   = 500.0 a.u.
  Frequency cutoff       = 2.0 a.u.
  La

### ▶️ Hands-on 2.3 Convergence of THC-GW w.r.t THC auxiliary basis
Modifying `thc_gw_conv.py` by increasing the `thresh` in the THC Coulomb Hamiltonian, and checking the convergence of the Hartree-Fock and correlation energies. 

In [ ]:
# --- Phase 2: Simulation (GW) ---
# hide timers at 2
coqui.set_verbosity(coqui_mpi, output_level=1)
gw_params = {
    "restart": False,
    "output": "gw",
    "niter": 1,
    "beta": 300,
    "lambda": 9000,
    "iaft_prec": "high",
}
coqui.run_gw(gw_params, h_int = thc)

# 🔹 CoQui HDF5 output structure 

The GW output of CoQui is stored in a single HDF5 archive with the following data structure:

####  Overview
```text
{prefix}.mbpt.h5
├── imaginary_fourier_transform/
├── scf/
└── system/
```

#### `imaginary_fourier_transform/`
```text
imaginary_fourier_transform/
├── beta                  # Inverse temperature β
├── lambda                # Dimensionsless paramter that control the accuracy
├── prec                  # Precision 
├── source                # Source of imaginary-axes grids: `dlr` or `ir` 
├── iwn_mesh/             # Fermionic Matsubara frequency mesh
└── tau_mesh/             # Imaginary-time mesh
```

#### `scf/`
Stores all quantities tracked during the self-consistent iterations of the GW or GW+EDMFT loop. Each iteration is saved as a subgroup iterN, with final_iter indicating the index of the final iteration.
```text
scf/
├── final_iter            # Final iteration index (e.g., 10)
├── iter0/
├── iter1/
│   ├── Dm_skij           # Density matrix (spin, kpoints, bands, bands)
│   ├── G_tskij           # Green's function G(τ, spin, kpoints, bands, bands)
│   ├── F_skij            # Static self-energy (spin, kpoints, bands, bands)
│   ├── Sigma_tskij       # Self-energy Σ(τ, spin, kpoints, bands, bands)
│   └── mu                # Chemical potential
├── iter2/
...
├── iter10/
```

### 🔹 Phase 3: Post-processing — why and what happens?

<figure style="text-align: center;"> <img src="./images/coqui_pproc.png" alt="Phase 3 of the GW workflow in CoQuí" width="60%"> <figcaption><em>Figure&nbsp;4:</em> Phase 3 of the GW workflow in CoQuí. </figcaption> </figure>

**Why post-process?**  
After the Dyson–SCF GW loop in Phase 2, we have G and Σ stored in `{output}.mbpt.h5`. These results are on the coarse mean-field k-mesh and in imaginary frequency. This k-mesh is usually too coarse to resolve fine features such as band dispersions, band gaps, or spectral peaks. In addition, core GW quantities are computed on the imaginary axis, requiring analytic continuation to the real axis for comparison with experiment. Therefore, Phase 3 interpolates them in k-space and performs analytic continuation to the real axis, producing physical observables that can be compared with experimental data. 

```python
# --- Phase 3: Post-processing ---
coqui_h5 = gw_params["output"]     # e.g., "gw"
wan_h5   = "data/qe_inputs/si/mlwf/si.mlwf.h5"  # path to your Wannier file
winter_params = {
    "outdir": "./",
    "prefix": coqui_h5,             # reads f"{prefix}.mbpt.h5"
    "iteration": -1,                # -1 = final SCF iteration
    "wannier_file": wan_h5,
    "ac_alg": "pade",               # analytic continuation algorithm
    "w_min": -0.4, "w_max": 0.4, "Nw": 10000,
    "bands_num_npoints": 100, 
    "kpath": """
      L 0.50 0.50 0.50
      G 0.00 0.00 0.00
      X 0.50 0.00 0.50
    """
}
spectral_interpolation(mf, winter_params)  # mf = your mean-field object
coqui_mpi.barrier()
```

The call `coqui.spectral_interpolation(...)` does the following things: 
- Read & interpolate (k-space):  
  Loads the selected GW iteration (often iteration=-1) from {output}.mbpt.h5 and uses your Wannier model to interpolate from the coarse QE mesh to a dense/high-symmetry k-path, evaluating G(k,iω$_{n}$).
- Analytic continuation (frequency):  
  Continues G(k,iω$_{n}$) → G(k, ω+iη) on the real axis (e.g., ac_alg="pade"), then computes the spectral function
  $$
    A(k,\omega) = -\frac{1}{\pi}\mathrm{Im Tr} G(k,\omega+i\zeta)
  $$

- Store for reuse:  
Writes the resulting ω-grid, k-path, and A(k,ω) back into the same `{output}.mbpt.h5`, so plotting tools (and future analyses) can read them directly. 

------
### Plotting helper
`plot_utils.spectral_plot` reads the post-processed datasets from {output}.mbpt.h5 and plots A(k,ω) along the high-symmetry path, with the Fermi level aligned at ω=0.

```python
# Plotting
if coqui_mpi.root():
    fig, ax = plt.subplots(1, figsize=(7,5.5), dpi=80)
    plot_utils.spectral_plot(ax, f"{coqui_h5}.mbpt.h5",
                             calc_type="mbpt", iteration=-1)
    ax.axhline(y=0, color="black", linestyle="-", linewidth=2.0, alpha=0.5)
    ax.set_ylim(-10.884, 10.884)
    ax.legend(loc=1, fontsize=16)
    plt.tight_layout()
    plt.savefig("gw_spectral.png", dpi=150)
```

>Practical tips:
> - Use iteration=-1 to always pick the final SCF state.
> - Tune eta, w_min/w_max, and Nw to balance resolution vs numerical stability.
> - Padé (Nfit) may need adjustment; too large can overfit, too small can smear features.

### 🔹 Hands-on 3 GW spectral functions v.s. DFT band structure
Modifying `thc_gw_spectral.py` to plot GW k-resolved spectral functions and compared them with the band structure of DFT. How do they differ from each other? 

- Plotting the band structure of GW at different iterations
- How does the spectral evolves from DFT to the converged GW solution?

###### TODO: Students can interpolate their own data calculated from 2x2x2, and then compare to the pre-computed data on 10x10x10 to see how Wannier interpolation is affected by the k-point size. 

**Questions**
- Note the relatively slow runtime for analytically continuing the GW solution to real-frequency axis compared to DFT+DMFT. Do you know why? 

> ⚠️ Note: For speed, some the data is pre-computed in the `data` folder. You can later re-run the calculations from scratch if desired.